In [18]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import model_inference


## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/28 17:39:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/28 17:39:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/10/28 17:39:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


### set up config

In [3]:
snapshot_date_str = "2024-01-01"
model_name = "credit_model_2024_09_01.pkl"

In [4]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


### load model artefact from model bank

In [5]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


### load feature store

In [6]:
# connect to feature store - clickstream
folder_path = "datamart/gold/feature_store_clickstream/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
feature_clickstream_store_sdf = spark.read.option("header", "true").parquet(*files_list)
feature_clickstream_sdf = feature_clickstream_store_sdf.filter((col("snapshot_date") == config["snapshot_date"]))

print("extracted feature_clickstream_sdf", feature_clickstream_sdf.count(), config["snapshot_date"])

feature_clickstream_pdf = feature_clickstream_sdf.toPandas()
feature_clickstream_pdf

extracted feature_clickstream_sdf 8974 2024-01-01 00:00:00


,Customer_ID,avg_fe_1,avg_fe_2,avg_fe_3,avg_fe_4,avg_fe_5,avg_fe_6,avg_fe_7,avg_fe_8,avg_fe_9,...,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20,snapshot_date
0,CUS_0x9154,142.666667,112.500000,61.333333,163.666667,83.500000,189.666667,97.333333,106.500000,85.500000,...,150.166667,79.166667,117.833333,150.000000,75.000000,113.000000,79.500000,125.166667,191.333333,2024-01-01
1,CUS_0x2c87,107.833333,104.000000,99.000000,126.000000,102.833333,114.833333,128.500000,106.500000,170.000000,...,122.000000,100.333333,123.833333,132.166667,153.500000,109.500000,62.333333,152.000000,124.000000,2024-01-01
2,CUS_0x333e,67.500000,115.000000,117.666667,69.166667,123.500000,222.500000,112.000000,193.333333,140.500000,...,122.000000,75.166667,83.166667,125.333333,109.666667,143.333333,137.500000,21.000000,50.500000,2024-01-01
3,CUS_0x6bbe,111.666667,83.833333,77.666667,119.000000,73.000000,127.333333,99.666667,107.833333,148.166667,...,122.166667,96.166667,76.500000,136.000000,137.166667,68.500000,114.833333,135.166667,111.500000,2024-01-01
4,CUS_0x995,77.000000,119.666667,102.000000,78.000000,51.333333,71.333333,120.166667,90.833333,153.500000,...,92.333333,135.666667,139.333333,88.666667,77.000000,101.500000,41.333333,121.333333,49.500000,2024-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,CUS_0x9f04,65.000000,57.333333,172.000000,73.333333,94.000000,118.333333,90.166667,126.333333,115.000000,...,150.166667,81.500000,119.666667,107.833333,93.500000,111.000000,66.500000,117.166667,38.333333,2024-01-01
8970,CUS_0x2ed0,146.000000,86.666667,96.833333,81.666667,182.166667,72.166667,93.166667,65.333333,96.166667,...,98.166667,125.000000,106.833333,49.166667,121.500000,98.166667,65.666667,79.833333,81.166667,2024-01-01
8971,CUS_0x68f8,74.833333,126.500000,136.000000,83.000000,88.666667,122.000000,84.833333,106.500000,194.333333,...,52.166667,92.333333,90.166667,86.666667,117.000000,124.166667,54.500000,135.500000,78.666667,2024-01-01
8972,CUS_0x2ac8,126.833333,137.666667,85.333333,108.500000,136.000000,137.666667,87.333333,138.833333,149.166667,...,111.333333,111.500000,93.833333,135.666667,67.333333,60.000000,82.000000,39.000000,79.666667,2024-01-01


In [7]:
# connect to feature store - financials
folder_path = "datamart/gold/feature_store_financials/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
feature_financial_store_sdf = spark.read.option("header", "true").parquet(*files_list)
feature_financial_sdf = feature_financial_store_sdf.filter((col("snapshot_date") == config["snapshot_date"]))

print("extracted feature_financial_sdf", feature_financial_sdf.count(), config["snapshot_date"])

feature_financial_pdf = feature_financial_sdf.toPandas()
feature_financial_pdf

extracted feature_financial_sdf 485 2024-01-01 00:00:00


,Customer_ID,snapshot_date,Num_Fin_Pdts,Debt_to_Salary,Loans_per_Credit_Item,Outstanding_Debt,Changed_Credit_Limit,Credit_History_Age_Month
0,CUS_0x102d,2024-01-01,9.0,0.089342,0.111111,648.359985,6.37,363
1,CUS_0x1051,2024-01-01,9.0,0.349741,0.111111,1000.440002,NaN,342
2,CUS_0x1269,2024-01-01,6.0,0.022200,0.750000,83.550003,14.11,207
3,CUS_0x1290,2024-01-01,14.0,0.113867,0.250000,83.160004,9.71,279
4,CUS_0x12d1,2024-01-01,10.0,0.419853,0.222222,691.530029,14.37,324
...,...,...,...,...,...,...,...,...
480,CUS_0xc68d,2024-01-01,13.0,0.988397,0.166667,1180.560059,9.61,191
481,CUS_0xc6d8,2024-01-01,10.0,0.036852,0.100000,369.359985,5.96,256
482,CUS_0xe8d,2024-01-01,16.0,0.918768,0.133333,1336.310059,9.85,162
483,CUS_0xf9e,2024-01-01,14.0,0.052316,0.153846,628.570007,1.65,368


In [8]:
# prepare data for modeling
features_pdf = feature_financial_sdf.join(feature_clickstream_sdf, on=["Customer_ID","snapshot_date"], how="left").toPandas()
features_pdf["Outstanding_Debt_log"] = np.log1p(features_pdf["Outstanding_Debt"])
features_pdf = features_pdf.drop('Outstanding_Debt', axis=1)
features_pdf

,Customer_ID,snapshot_date,Num_Fin_Pdts,Debt_to_Salary,Loans_per_Credit_Item,Changed_Credit_Limit,Credit_History_Age_Month,avg_fe_1,avg_fe_2,avg_fe_3,...,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20,Outstanding_Debt_log
0,CUS_0x2059,2024-01-01,12.0,0.012866,0.181818,NaN,403,75.500000,155.333333,80.666667,...,158.000000,139.000000,113.000000,70.166667,40.166667,64.833333,95.666667,93.166667,86.333333,4.865070
1,CUS_0x5bb7,2024-01-01,8.0,0.058452,0.800000,8.34,312,94.500000,49.333333,157.833333,...,115.666667,114.833333,82.000000,66.166667,95.500000,108.666667,133.166667,133.000000,120.166667,6.292661
2,CUS_0x6945,2024-01-01,12.0,0.111311,0.300000,14.35,114,138.833333,118.333333,80.166667,...,68.000000,119.333333,114.500000,112.833333,110.666667,123.833333,120.000000,97.166667,150.500000,5.154043
3,CUS_0x9886,2024-01-01,11.0,0.066529,0.333333,8.37,243,103.500000,89.166667,124.833333,...,104.500000,129.666667,122.666667,122.833333,157.833333,105.500000,177.666667,46.666667,89.833333,6.828593
4,CUS_0x19b5,2024-01-01,13.0,0.778182,0.076923,7.89,214,114.500000,52.000000,185.000000,...,112.500000,75.833333,39.000000,87.500000,49.666667,140.500000,166.333333,122.166667,137.166667,6.601733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,CUS_0x9942,2024-01-01,10.0,0.116416,0.000000,14.17,206,204.333333,112.833333,90.666667,...,93.666667,124.166667,222.333333,110.333333,139.500000,102.666667,93.833333,117.500000,58.500000,6.321523
481,CUS_0x1718,2024-01-01,10.0,0.021634,0.100000,13.94,248,194.166667,162.000000,90.166667,...,85.333333,112.500000,99.500000,88.500000,100.333333,99.333333,173.833333,69.833333,97.333333,3.935544
482,CUS_0x2d14,2024-01-01,5.0,0.596470,0.000000,8.42,302,116.666667,139.166667,49.666667,...,108.500000,134.166667,103.000000,87.000000,123.500000,155.833333,130.500000,169.500000,170.000000,6.861450
483,CUS_0x7a12,2024-01-01,10.0,0.179513,0.000000,8.35,201,82.333333,137.000000,212.500000,...,113.500000,64.166667,117.666667,89.500000,82.166667,100.166667,84.666667,99.166667,94.000000,7.056649


### preprocess data for modeling

In [11]:
# prepare X_inference
feature_cols = ['avg_fe_1',
 'avg_fe_2',
 'avg_fe_3',
 'avg_fe_4',
 'avg_fe_5',
 'avg_fe_6',
 'avg_fe_7',
 'avg_fe_8',
 'avg_fe_9',
 'avg_fe_10',
 'avg_fe_11',
 'avg_fe_12',
 'avg_fe_13',
 'avg_fe_14',
 'avg_fe_15',
 'avg_fe_16',
 'avg_fe_17',
 'avg_fe_18',
 'avg_fe_19',
 'avg_fe_20',
 'Num_Fin_Pdts',
 'Debt_to_Salary',
 'Loans_per_Credit_Item',
 'Changed_Credit_Limit',
 'Credit_History_Age_Month',
 'Outstanding_Debt_log']

X_inference = features_pdf[feature_cols]

# apply transformer - standard scaler
transformer_stdscaler = model_artefact["preprocessing_transformers"]["stdscaler"]
X_inference = transformer_stdscaler.transform(X_inference)

print('X_inference', X_inference.shape[0])
X_inference

X_inference 485


array([[-0.7478576 ,  0.941709  , -0.64393788, ...,         nan,
         1.80538237, -1.86066615],
       [-0.34412053, -1.2976406 ,  0.92832589, ..., -0.33308049,
         0.88763974, -0.5228881 ],
       [ 0.59793264,  0.16004923, -0.65412533, ...,  0.57385019,
        -1.10920687, -1.58987329],
       ...,
       [ 0.12690605,  0.60017298, -1.27556004, ..., -0.32100822,
         0.7867889 ,  0.01011655],
       [-0.60265392,  0.55440011,  2.04215421, ..., -0.33157142,
        -0.23180457,  0.19303556],
       [ 1.72060502,  0.5579211 , -0.87824933, ..., -0.34967998,
        -1.49244006,  0.25010126]], shape=(485, 26))

### model prediction inference

In [12]:
# load model
model = model_artefact["model"]

# predict model
y_inference = model.predict_proba(X_inference)[:, 1]

# prepare output
y_inference_pdf = features_pdf[["Customer_ID","snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference
y_inference_pdf

,Customer_ID,snapshot_date,model_name,model_predictions
0,CUS_0x2059,2024-01-01,credit_model_2024_09_01.pkl,0.123797
1,CUS_0x5bb7,2024-01-01,credit_model_2024_09_01.pkl,0.234930
2,CUS_0x6945,2024-01-01,credit_model_2024_09_01.pkl,0.154780
3,CUS_0x9886,2024-01-01,credit_model_2024_09_01.pkl,0.181825
4,CUS_0x19b5,2024-01-01,credit_model_2024_09_01.pkl,0.530109
...,...,...,...,...
480,CUS_0x9942,2024-01-01,credit_model_2024_09_01.pkl,0.146063
481,CUS_0x1718,2024-01-01,credit_model_2024_09_01.pkl,0.234397
482,CUS_0x2d14,2024-01-01,credit_model_2024_09_01.pkl,0.155304
483,CUS_0x7a12,2024-01-01,credit_model_2024_09_01.pkl,0.167580


### save model inference to datamart gold table

In [13]:
# create bronze datalake
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_01_01.parquet


### backfill

In [14]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [15]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [19]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


extracted feature_clickstream_sdf 8974 2023-01-01 00:00:00


extracted feature_financial_sdf 530 2023-01-01 00:00:00


X_inference 530
datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_01_01.parquet


---completed job---


2023-02-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 2, 1, 0, 0),
 'snapshot_date_str': '2023-02-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


...

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_11_01.parquet


---completed job---


2024-12-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 12, 1, 0, 0),
 'snapshot_date_str': '2024-12-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


extracted feature_clickstream_sdf 8974 2024-12-01 00:00:00


extracted feature_financial_sdf 515 2024-12-01 00:00:00


X_inference 515
datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_12_01.parquet


---completed job---


